# 콩당콩당 BGE-M3 Embedding LoRA 파인튜닝 파이프라인

본 노트북은 만성 신장병(CKD) 및 당뇨(DM) 맞춤형 질문-답변 및 코퍼스 매칭을 고도화하기 위해 `BAAI/bge-m3` 기반 임베딩 모델을 PEFT(LoRA) 기법으로 미세 조정하는 실습 코드입니다.

In [ ]:
# 1. 필요 패키지 설치
!pip install -q transformers peft datasets accelerate sentence-transformers yaml pyarrow

In [ ]:
# 2. 패키지 로드 및 설정 로드
import os
import yaml
import torch
from transformers import AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Config loaded successfully:", config)

In [ ]:
# 3. 데이터셋 로드
# 데이터셋 구조: {'anchor': '질문', 'positive': '관련 초록 또는 가이드라인', 'negative': '유사하나 무관한 초록'}
try:
    dataset = load_dataset('json', data_files='../../data/ckd_dm_qa_pairs/train.json')
    print("데이터셋 로드 성공! 크기:", len(dataset['train']))
except Exception as e:
    print("데이터셋 파일을 확인해 주세요. 로컬 가짜 데이터 생성 중...")
    # 데모용 데이터셋 가상 생성
    from datasets import Dataset
    dummy_data = {
        "anchor": ["만성 신부전 3기 칼륨 식이요법 알려줘", "당뇨 신증 미세알부민 조기검사 주기"],
        "positive": ["만성 신부전 3기는 칼륨 배설 장애가 오므로 바나나, 토마토 등의 고칼륨 식이 제한이 필수적입니다.", "당뇨 환자는 신장 합병증 예방을 위해 매년 최소 1회 미세알부민뇨 요검사를 권장합니다."],
        "negative": ["나트륨을 줄이고 유산소 운동을 하십시오.", "당뇨 환자의 탄수화물 제한 식이 가이드라인 요약입니다."]
    }
    dataset = Dataset.from_dict(dummy_data)
    print("가상 데이터셋 준비 완료.")

In [ ]:
# 4. BGE-M3 모델 및 토크나이저 초기화
model_name = config['model']['base_model_name_or_path']
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

# LoRA 설정 주입
peft_config = LoraConfig(
    r=config['lora']['r'],
    lora_alpha=config['lora']['lora_alpha'],
    lora_dropout=config['lora']['lora_dropout'],
    target_modules=config['lora']['target_modules'],
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. 대조 학습(Contrastive Loss) 및 SentenceTransformers 파인튜닝 루프 정의
# 학습 완료 후 가중치 모델 저장
output_dir = "./models/bge-m3-lora"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"파인튜닝 완료 및 PEFT 가중치 저장 완료: {output_dir}")